In [40]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow multiple libiomp5md
os.environ["OMP_NUM_THREADS"] = "1"           # keep OpenMP under control

In [41]:
path = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0"
df = pd.read_csv(path + "//labels_mbrset.csv")

In [42]:
df.isna().sum()

patient                           0
age                               0
sex                               0
dm_time                          56
insulin                          48
insulin_time                   4148
oraltreatment_dm                 44
systemic_hypertension            44
insurance                        48
educational_level                52
alcohol_consumption              76
smoking                          88
obesity                          76
vascular_disease                 76
acute_myocardial_infarction      84
nephropathy                      80
neuropathy                       76
diabetic_foot                   108
file                              0
laterality                        0
final_artifacts                   0
final_quality                     0
final_icdr                      280
final_edema                     265
dtype: int64

In [43]:
df.columns

Index(['patient', 'age', 'sex', 'dm_time', 'insulin', 'insulin_time',
       'oraltreatment_dm', 'systemic_hypertension', 'insurance',
       'educational_level', 'alcohol_consumption', 'smoking', 'obesity',
       'vascular_disease', 'acute_myocardial_infarction', 'nephropathy',
       'neuropathy', 'diabetic_foot', 'file', 'laterality', 'final_artifacts',
       'final_quality', 'final_icdr', 'final_edema'],
      dtype='object')

In [44]:
#Extracting required columns

df = df[["patient", "file", "final_icdr", "final_quality", "laterality"]]

In [45]:
df.head()

,patient,file,final_icdr,final_quality,laterality
0,1,1.1.jpg,4.0,yes,right
1,1,1.2.jpg,4.0,yes,right
2,1,1.3.jpg,4.0,yes,left
3,1,1.4.jpg,4.0,yes,left
4,10,10.1.jpg,0.0,yes,right


In [46]:
df.shape

(5164, 5)

In [47]:
df["final_quality"].value_counts()

final_quality
yes    4872
no      292
Name: count, dtype: int64

In [48]:
# final_quality: yes -> 1, no -> 0
df["final_quality"] = df["final_quality"].map({"yes": 1, "no": 0})

# laterality: right -> 1, left -> 0
df["laterality"] = df["laterality"].map({"right": 1, "left": 0})


In [49]:
# propagate ICDR within each (patient, eye)

df["final_icdr"] = (
    df.groupby(["patient", "laterality"])["final_icdr"]
      .transform(lambda x: x.ffill().bfill())
)

In [50]:
print(df[df["final_icdr"].isna()])

      patient        file  final_icdr  final_quality  laterality
20       1002  1002.1.jpg         NaN              0           1
21       1002  1002.2.jpg         NaN              0           1
104      1022  1022.1.jpg         NaN              0           1
105      1022  1022.2.jpg         NaN              0           1
372      1089  1089.1.jpg         NaN              0           1
...       ...         ...         ...            ...         ...
5033      968   968.2.jpg         NaN              0           1
5046      970   970.3.jpg         NaN              0           0
5047      970   970.4.jpg         NaN              0           0
5158      998   998.3.jpg         NaN              0           0
5159      998   998.4.jpg         NaN              0           0

[118 rows x 5 columns]


In [51]:
df.shape

(5164, 5)

In [52]:
df.isna().sum()

patient            0
file               0
final_icdr       118
final_quality      0
laterality         0
dtype: int64

In [53]:
df = df.dropna()

In [54]:
df.shape

(5046, 5)

In [55]:
df.isna().sum()

patient          0
file             0
final_icdr       0
final_quality    0
laterality       0
dtype: int64

In [56]:

df["final_icdr"] = df["final_icdr"].apply(lambda x: 0 if x == 0 else 1)

In [57]:
# count images per patient-eye
patient_eye_counts = df.groupby(["patient", "laterality"]).size().unstack(fill_value=0)

# total patients
total_patients = len(patient_eye_counts)

# patients where BOTH eyes have exactly 2 images
both_exactly_2 = (patient_eye_counts[0] == 2) & (patient_eye_counts[1] == 2)

num_both_exactly_2 = both_exactly_2.sum()
percent_both_exactly_2 = num_both_exactly_2 / total_patients * 100

print("Total patients:", total_patients)
print("Patients where BOTH eyes have exactly 2 images:", num_both_exactly_2,
      f"({percent_both_exactly_2:.2f}%)")

# optional: print the actual patients
print(patient_eye_counts[both_exactly_2])

Total patients: 1287
Patients where BOTH eyes have exactly 2 images: 1234 (95.88%)
laterality  0  1
patient         
1           2  2
2           2  2
3           2  2
4           2  2
5           2  2
...        .. ..
1329        2  2
1330        2  2
1332        2  2
1334        2  2
1335        2  2

[1234 rows x 2 columns]


In [39]:
# keep patients with exactly 2 images per eye (left + right)
valid_patients = df.groupby(["patient", "laterality"]).size().unstack(fill_value=0)
print(len(valid_patients))
valid_patients = valid_patients[(valid_patients[0] == 2) & (valid_patients[1] == 2)].index
print(len(valid_patients))
df = df[df["patient"].isin(valid_patients)]

1234
1234


In [37]:
df.shape


(4936, 5)

In [19]:
df.head(50)

,patient,file,final_icdr,final_quality,laterality
0,1,1.1.jpg,1,1,1
1,1,1.2.jpg,1,1,1
2,1,1.3.jpg,1,1,0
3,1,1.4.jpg,1,1,0
4,10,10.1.jpg,0,1,1
5,10,10.2.jpg,0,0,1
6,10,10.3.jpg,0,1,0
7,10,10.4.jpg,0,1,0
8,100,100.1.jpg,0,1,1
9,100,100.2.jpg,0,1,1


In [20]:
df["final_icdr"] = (
    df.groupby(["patient", "laterality"])["final_icdr"]
      .transform("max")
)

In [21]:
df.shape

(4936, 5)

In [22]:
# One label per patient - take max label just for stratified split 
patient_df = (
    df
    .groupby("patient", as_index=False)
    .agg(final_icdr=("final_icdr", "max"))
)


# 70 / 15 / 15 stratified split
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.30,
    stratify=patient_df["final_icdr"],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    stratify=temp_patients["final_icdr"],
    random_state=42
)

# Map patients back to images
train_df = df[df["patient"].isin(train_patients["patient"])]
val_df   = df[df["patient"].isin(val_patients["patient"])]
test_df  = df[df["patient"].isin(test_patients["patient"])]

# Image counts
print("Image counts:")
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

print("Patient counts:")
print("train:", len(train_patients["patient"]))
print("val:", len(val_patients["patient"]))
print("test:", len(test_patients["patient"]))

# Patient-level DR ratio
def print_balance(name, patient_ids):
    subset = patient_df[patient_df["patient"].isin(patient_ids)]
    print(f"{name} DR ratio:", subset["final_icdr"].mean())

print_balance("Train", train_patients["patient"])
print_balance("Val", val_patients["patient"])
print_balance("Test", test_patients["patient"])

Image counts:
Train: 3452
Val:   740
Test:  744
Patient counts:
train: 863
val: 185
test: 186
Train DR ratio: 0.31170336037079954
Val DR ratio: 0.3081081081081081
Test DR ratio: 0.3118279569892473


In [26]:
df.shape

(4936, 5)

In [34]:
len(df[df["final_icdr"]==0])

3674

In [35]:
len(df[df["final_icdr"]==1])

1262

In [37]:
1262/4936

0.25567260940032416

In [27]:
4936/4

1234.0

In [23]:
train_df.shape

(3452, 5)

In [24]:
val_df.shape

(740, 5)

In [25]:
test_df.shape

(744, 5)

In [48]:
data_dir = r'C:\\Users\\preet\\Documents\\mBRSET\\mBRSET_image_quality\\'
train_df.to_pickle(data_dir + "data\\mbrset_icdr_quality_524_train_full.pkl")
val_df.to_pickle(data_dir + "data\\mbrset_icdr_quality_524_val_full.pkl")
test_df.to_pickle(data_dir + "data\\mbrset_icdr_quality_524_test_full.pkl")